## XGBoost

O [XGBoost](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html) (Extreme Gradient Boosting) é um modelo baseado em árvores que utiliza a técnica de boosting, onde múltiplos modelos fracos são combinados sequencialmente para formar um modelo forte.

A cada nova árvore, o modelo tenta corrigir os erros das árvores anteriores, melhorando progressivamente o desempenho.

### Hiperparâmetros utilizados

- **n_estimators**: número de árvores no modelo
  - Mais árvores → melhor aprendizado (até certo limite)

- **learning_rate**: taxa de aprendizado
  - Valores menores tornam o aprendizado mais lento, porém mais robusto

- **max_depth**: profundidade das árvores
  - Controla a complexidade do modelo

Estratégia

Foi utilizado GridSearchCV com validação cruzada (5 folds) e métrica ROC AUC para encontrar a melhor combinação de hiperparâmetros.

In [1]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []

for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote,
        )

        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

        steps = []
        
        if use_smote:
            steps.append(('smote', SMOTE(random_state=42)))
        steps.append(('xgb', XGBClassifier(random_state=42)))

        model = Pipeline(steps)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "XGBoost",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "timestamp": pd.Timestamp.now()
        })


df_result = pd.DataFrame(results)

save_results(df_result)

display(df_result)



🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,timestamp
0,XGBoost,sem_submodalidade,False,0.930321,0.864498,0.847728,67,2026-04-09 21:01:38.604250
1,XGBoost,sem_submodalidade,True,0.926829,0.867960,0.847673,67,2026-04-09 21:01:48.280996
2,XGBoost,submodalidade_agrupada,False,0.943875,0.880579,0.864592,97,2026-04-09 21:01:52.986582
3,XGBoost,submodalidade_agrupada,True,0.942460,0.884649,0.865943,97,2026-04-09 21:02:03.817310
4,XGBoost,submodalidade_engineered,False,0.930321,0.864498,0.847728,67,2026-04-09 21:02:08.065526
5,XGBoost,submodalidade_engineered,True,0.926829,0.867960,0.847673,67,2026-04-09 21:02:17.975626


## GRIDSEARCH

In [3]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

param_grid_xgb = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "xgb__n_estimators": [200, 400],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__max_depth": [3, 5],
    "xgb__subsample": [0.8, 1],
    "xgb__colsample_bytree": [0.8, 1]
}

grid_xgb = GridSearchCV(
    estimator=Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('xgb', XGBClassifier(eval_metric="logloss",
        random_state=42,
        scale_pos_weight=scale_pos_weight,))
    ]),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Best parameters:", grid_xgb.best_params_)
print("Best ROC AUC (CV):", grid_xgb.best_score_)


#? BEST MODEL TEST EVALUATION

best_xgb = grid_xgb.best_estimator_

y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_results = []

cv_results = pd.DataFrame(grid_xgb.cv_results_)
cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

for smote_val in [True, False]:
    sub_results = cv_results[cv_results['smote_used'] == smote_val]
    if not sub_results.empty:
        best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
        params = best_row['params']
        
        tuning_results.append({
            "model": "XGBoost",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "tuning_cv",
            "roc_auc": best_row['mean_test_score'],
            "f1": None,
            "accuracy": None,
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })
        
        # Re-fit the best model for this group to get test metrics
        best_model_group = grid_xgb.estimator.set_params(**params)
        best_model_group.fit(X_train, y_train)
        
        y_pred = best_model_group.predict(X_test)
        y_proba = best_model_group.predict_proba(X_test)[:, 1]
        
        tuning_results.append({
            "model": "XGBoost",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "test",
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))

Best parameters: {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 400, 'subsample': 0.8}
Best ROC AUC (CV): 0.9303426157742501


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,XGBoost,sem_submodalidade,False,test,0.930373,0.864766,0.848184,"{'colsample_bytree': 1, 'learning_rate': 0.1, ...",2026-04-09 21:04:33.153976
